# Laboratorio 02 — Notebook Genérico de Ingestión Bronze

**Semana:** 04 | **Actividad de referencia:** Actividad 02  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Diseña un notebook de ingestión Bronze **genérico y reutilizable** que acepte widgets para el formato de entrada, la ruta del archivo, el nombre de la tabla destino y el modo de escritura. El notebook debe poder ingestar tu dataset propio en Bronze **sin cambiar el código**, solo cambiando los widgets.

## Parte 1 — Descripción del dataset y diseño del notebook

1. **Nombre, fuente y URL** del dataset.
2. **Formato del archivo:** CSV, JSON, Parquet u otro. ¿Por qué ese formato?
3. **Esquema esperado:** Lista las columnas y sus tipos. ¿Usarás `inferSchema` o definirás el esquema explícito? ¿Por qué?
4. **Diseño de la interfaz de widgets:** ¿Qué parámetros debería recibir el notebook para ser verdaderamente genérico?

**Escribe tu respuesta aquí:**

## Parte 2 — Definición de Widgets

In [ ]:
dbutils.widgets.removeAll()

# Formato de entrada
dbutils.widgets.dropdown(
    name         = "formato",
    defaultValue = "csv",
    choices      = ["csv", "json", "parquet", "delta"],
    label        = "Formato del archivo fuente"
)

# Ruta del archivo en el volumen
dbutils.widgets.text(
    name         = "ruta_origen",
    defaultValue = "/Volumes/workspace/default/week_4/tu_archivo.csv",
    label        = "Ruta completa al archivo fuente"
)

# Nombre de la tabla Delta destino
dbutils.widgets.text(
    name         = "tabla_destino",
    defaultValue = "workspace.default.bronze_mi_dataset",
    label        = "Tabla Delta destino (catalog.schema.tabla)"
)

# Modo de escritura
dbutils.widgets.dropdown(
    name         = "modo_escritura",
    defaultValue = "overwrite",
    choices      = ["overwrite", "append", "ignore", "errorifexists"],
    label        = "Modo de escritura Delta"
)

# Inferir esquema automáticamente
dbutils.widgets.dropdown(
    name         = "inferir_schema",
    defaultValue = "true",
    choices      = ["true", "false"],
    label        = "Inferir esquema automáticamente"
)

print("✓ Widgets definidos")

## Parte 3 — Leer parámetros y preparar la lectura

In [ ]:
formato       = dbutils.widgets.get("formato")
ruta_origen   = dbutils.widgets.get("ruta_origen")
tabla_destino = dbutils.widgets.get("tabla_destino")
modo          = dbutils.widgets.get("modo_escritura")
inferir       = dbutils.widgets.get("inferir_schema") == "true"

print(f"  formato       = {formato}")
print(f"  ruta_origen   = {ruta_origen}")
print(f"  tabla_destino = {tabla_destino}")
print(f"  modo          = {modo}")
print(f"  inferir       = {inferir}")

## Parte 4 — Ingestión genérica multiformato

In [ ]:
# Opciones de lectura por formato
opciones_por_formato = {
    "csv":     {"header": "true",  "inferSchema": str(inferir).lower()},
    "json":    {"multiLine": "true"},
    "parquet": {},
    "delta":   {}
}

reader = spark.read.format(formato)
for k, v in opciones_por_formato.get(formato, {}).items():
    reader = reader.option(k, v)

df_bronze = reader.load(ruta_origen)
print(f"✓ Lectura exitosa: {df_bronze.count():,} filas | {len(df_bronze.columns)} columnas")
df_bronze.printSchema()

## Parte 5 — Perfil del dataset antes de escribir (quality gate)

In [ ]:
from pyspark.sql import functions as F

total = df_bronze.count()

# Porcentaje de nulos por columna
nulos = df_bronze.select([
    F.round(
        F.sum(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), 1).otherwise(0))
        * 100.0 / total, 1
    ).alias(f"{c}_pct_nulos")
    for c in df_bronze.columns
])
nulos.show(truncate=False)

In [ ]:
# Estadísticas descriptivas
df_bronze.describe().show(truncate=False)

**Calidad del dato antes de Bronze:**  
¿Hay columnas con >30% de nulos que deberías documentar? ¿Encontraste tipos de dato incorrectos (ej. número como string)?  
Documenta aquí los hallazgos para que queden en el historial del notebook.

## Parte 6 — Añadir metadatos de auditoría y escribir en Bronze

In [ ]:
from datetime import datetime

# Añadir columnas de auditoría (patrón estándar de capa Bronze)
df_con_meta = df_bronze \
    .withColumn("_ingest_timestamp", F.current_timestamp()) \
    .withColumn("_source_file",      F.lit(ruta_origen)) \
    .withColumn("_source_format",    F.lit(formato))

# Escribir como tabla Delta
df_con_meta.write \
    .format("delta") \
    .mode(modo) \
    .saveAsTable(tabla_destino)

print(f"✓ Tabla Bronze creada: {tabla_destino}")
print(f"  Filas escritas: {df_con_meta.count():,}")
print(f"  Modo:           {modo}")

In [ ]:
# Verificar el historial de la tabla Delta creada
spark.sql(f"DESCRIBE HISTORY {tabla_destino}").show(5, truncate=False)

## Parte 7 — Probar el notebook con un segundo dataset

Cambia los widgets (sin modificar el código) para apuntar a un segundo archivo diferente al original, con modo `append` o `overwrite`. Luego ejecuta el notebook desde la Parte 3 y verifica el resultado.

In [ ]:
# Verificar que la tabla Delta contiene los datos del segundo dataset (o el acumulado si usaste append)
spark.table(tabla_destino).groupBy("_source_file").count().orderBy("_source_file").show(truncate=False)

**Preguntas de negocio:**
1. ¿Cuántas filas provienen de cada archivo fuente?
2. ¿El modo `append` duplicó algún registro o funcionó correctamente?
3. ¿Qué ventaja tiene agregar `_ingest_timestamp` y `_source_file` en Bronze?

In [ ]:
# Exportar el resultado como indicador para el notebook padre
filas_escritas = spark.table(tabla_destino).count()
dbutils.notebook.exit(str(filas_escritas))

## Parte 8 — Reflexión final

1. ¿Qué diferencia hay entre `inferSchema=True` y definir el esquema explícitamente? ¿Cuál es más seguro en producción y por qué?
2. ¿Qué sucede si usas modo `errorifexists` y la tabla ya existe? ¿Cómo lo manejarías en un pipeline programado?
3. ¿Por qué la capa Bronze debería guardar los datos sin transformaciones y con columnas de auditoría?
4. ¿Cómo extenderías este notebook para soportar también archivos en S3 o Azure Data Lake?

---

## Entrega en Git

```bash
git add semana_04/laboratorios/lab_02_bronze_generico.ipynb
git commit -m "lab: semana04 lab02 bronze generico multiformato <nombre-dataset> - <tu-nombre>"
git push origin feature/semana04-metadata-<tu-nombre>
```